## Embed the Wikipedia Movie Plots dataset and then based on a query find the most similar movie and then a Little Qwen-Model to be a expert in movies using some Form of RAG


In [80]:
# Hugging-Face
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

# Torch
import torch

# Kaggle import libraries
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [24]:
file_path = "wiki_movie_plots_deduped.csv"

# Load the latest version
hf_dataset = kagglehub.load_dataset(
    KaggleDatasetAdapter.HUGGING_FACE,
    "jrobischon/wikipedia-movie-plots",
    file_path,
)

print("Hugging Face Dataset:", hf_dataset)

/var/folders/d8/k55r2d1s5k78lrh9j8l3lfzw0000gn/T/ipykernel_46483/2763955198.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  hf_dataset = kagglehub.load_dataset(


Hugging Face Dataset: Dataset({
    features: ['Release Year', 'Title', 'Origin/Ethnicity', 'Director', 'Cast', 'Genre', 'Wiki Page', 'Plot'],
    num_rows: 34886
})


In [7]:
hf_dataset["Plot"][1]

"The moon, painted with a smiling face hangs over a park at night. A young couple walking past a fence learn on a railing and look up. The moon smiles. They embrace, and the moon's smile gets bigger. They then sit down on a bench by a tree. The moon's view is blocked, causing him to frown. In the last scene, the man fans the woman with his hat because the moon has left the sky and is perched over her shoulder to see everything better."

In [25]:
hf_dataset.remove_columns(
    [
        "Release Year",
        "Origin/Ethnicity",
        "Director",
        "Cast",
        "Genre",
        "Wiki Page",
    ]
)

Dataset({
    features: ['Title', 'Plot'],
    num_rows: 34886
})

In [29]:
model_id = "all-MiniLM-L6-v2"
device = "mps"

In [31]:
model = SentenceTransformer(model_id)
# embeddings = model.encode(hf_dataset['Plot'])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
embeddings = model.encode(hf_dataset["Plot"])

In [ ]:
pairs = list(zip(hf_dataset["Title"], hf_dataset["Plot"], embeddings))

In [49]:
pairs[0][0], pairs[0][1]

('Kansas Saloon Smashers',
 "A bartender is working at a saloon, serving drinks to customers. After he fills a stereotypically Irish man's bucket with beer, Carrie Nation and her followers burst inside. They assault the Irish man, pulling his hat over his eyes and then dumping the beer over his head. The group then begin wrecking the bar, smashing the fixtures, mirrors, and breaking the cash register. The bartender then sprays seltzer water in Nation's face before a group of policemen appear and order everybody to leave.[1]")

In [74]:
def find_similar_movie(sentence: str) -> list[tuple]:
    sentence_embedding = model.encode(sentence)
    similarities_score, index_score = torch.topk(
        model.similarity(sentence_embedding, embeddings), k=5
    )
    return [(pairs[i.item()][0], pairs[i.item()][1]) for i in index_score.squeeze(0)]


In [76]:
# sentence = "movies with actions shooting and cars"
# sentence_embedding = model.encode(sentence)
# similarities_score, index_score = torch.topk(
#     model.similarity(sentence_embedding, embeddings), k=5
# )
# [(pairs[i.item()][0], pairs[i.item()][1]) for i in index_score.squeeze(0)]

In [77]:
find_similar_movie("A film with dragons and fantasy")

[('The Mad Monk',
  'The movie starts out with all of the gods in heaven complaining to the Jade Emperor about the malicious practical jokes played on them by Dragon Fighter Lohan. The Emperor summons Dragon\'s sworn-brother, Tiger Fighter Lohan (Ng Man Tat), in order to find Dragon. Dragon (Stephen Chow) eventually appears and rebuke the various gods for their horrible judgments on mankind. He insisted he can do a better job and is banished to be reincarnated into an animal by the Jade Emperor. The Bodhisattva Guan Yin (Anita Mui) intervene and issues Dragon a challenge. If he can change the fates of three people—a beggar, a prostitute, and a villain—doomed to nine incarnations as their current position in life within three heavenly days (thirty years), without heavenly powers, he will be upgraded in heavenly status. If he fails, he will be downgraded from an arhat to an animal. The Bodhisattva gives him a magical fan that can only be used three times a day for sleight-of-hand-like ma

In [ ]:
class MovieExpertBot:
    def __init__(self, model_id):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, device_map="auto", use_kernels=True
        )
        self.instruction = "You are an expert and a highly proclaimed film and tv-series critique and you should know how to recommend movies"

    def send_message_to_model(self, message):
        prompt = message + self.instruction
        inputs = self.tokenizer(prompt, return_tensors="pt")
        generate_ids = self.model.generate(inputs.input_ids, max_length=256)
        return answer = self.tokenizer.batch_decode(
            generate_ids, skip_special_tokens=True, clean_up_tokenization_sapces=False
        )[0]



: 

In [ ]:
movie_exp_model = MovieExpertBot("Qwen/Qwen3.5-4B")
answer = movie_exp_model.send_message_to_model("Which is the best action movie ? ")
answer

In [ ]:
prompt = "Hey, are you conscious? Can you talk to me?"
inputs = tokenizer(prompt, return_tensors="pt")

# Generate
generate_ids = model.generate(inputs.input_ids, max_length=30)
tokenizer.batch_decode(
    generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]
